In [ ]:
from utils import *
import k3d
st = time.time()

In [ ]:
path="../../DATA/registration_test/FB_100030____FB,2817673489_full/study_0c1e55cc/"
path_subfolder1=path+"MR7_bb3b1a0b"#"MR3_33a7c6a2"#"MR4_c61744e9"#"MR3_33a7c6a2"
path_subfolder2=path+"MR5_2e0c276a"
path_subfolder3=path+"MR3_33a7c6a2"

In [ ]:
# read 2 series (sagittal / coronal / axial)
sag_image=serie_reader(path_subfolder1)
cor_image = serie_reader(path_subfolder2)
ax_image= serie_reader(path_subfolder3)

vType= sag_image.GetPixelIDTypeAsString()

In [ ]:
# get spacing of each volume (dependent of orientation)
spacing_sag=get_spacing(sag_image)
spacing_cor=get_spacing(cor_image)
spacing_ax=get_spacing(ax_image)

print(spacing_sag, spacing_cor, spacing_ax)

In [ ]:
# get direction cosin matrix of each volume (dependent of orientation)
dir_sag = get_direction(sag_image)
dir_cor = get_direction(cor_image)
dir_ax = get_direction(ax_image)

In [ ]:
#dot product between spacing and Direct Cosine Matrix
A_sag=np.dot(dir_sag, spacing_sag)
A_cor=np.dot(dir_cor, spacing_cor)
A_ax=np.dot(dir_ax, spacing_ax)

In [ ]:
#get origin from axial, sag, cor series
origin_sag= np.array(sag_image.GetOrigin())
origin_cor= np.array(cor_image.GetOrigin())
origin_ax = np.array(ax_image.GetOrigin())
print(origin_cor, origin_ax, origin_sag)

In [ ]:
#get intensities from each series
m_sag=sitk.GetArrayFromImage(sag_image)
m_cor=sitk.GetArrayFromImage(cor_image)
m_ax=sitk.GetArrayFromImage(ax_image)

print(ax_image.GetSize(), m_ax.shape)

In [ ]:
itk_view = itk_view_from_simpleitk(m_ax, ax_image.GetSpacing(), dir_ax, origin_ax)
itkwidgets.view(itk_view)

In [ ]:
# calcul physical coordinates of each volume
print("sag \n")
xspa_sag=calcul_physicsCoo2(origin_sag, A_sag, m_sag)
print("cor \n")
xspa_cor=calcul_physicsCoo2(origin_cor, A_cor, m_cor)
print("ax \n")
xspa_ax=calcul_physicsCoo2(origin_ax, A_ax, m_ax)

In [ ]:
#concat all points from each volume
sag_d= pd.DataFrame(xspa_sag, columns=('x', 'y', 'z'))
cor_d= pd.DataFrame(xspa_cor, columns=('x', 'y', 'z'))
ax_d= pd.DataFrame(xspa_ax, columns=('x', 'y', 'z'))

datas = [sag_d, cor_d, ax_d]
datas_combined = pd.concat(datas)
print(datas_combined.head())

newres=np.min(np.asarray([sag_image.GetSpacing(), cor_image.GetSpacing(), ax_image.GetSpacing()]))
print(newres)

In [ ]:
#determine min coordinates, max coordinates

min_xyz=np.array([datas_combined['x'].min(), datas_combined['y'].min(), datas_combined['z'].min()]) 
max_xyz=np.array([datas_combined['x'].max(), datas_combined['y'].max(), datas_combined['z'].max()])
print(min_xyz.dtype, max_xyz)
new_origin, new_spacing, dir_new = define_param(newres, min_xyz, max_xyz)

sizeX=ceil((datas_combined['x'].max()-datas_combined['x'].min())/newres)
sizeY=ceil((datas_combined['y'].max()-datas_combined['y'].min())/newres)
sizeZ=ceil((datas_combined['z'].max()-datas_combined['z'].min())/newres)
print(newres, new_origin, new_spacing, dir_new, sizeX, sizeY, sizeZ)

In [ ]:
print(sag_d.x.min(), sag_d.y.min(), sag_d.z.min(), origin_sag)
print(m_sag.shape[0])

In [17]:
plt_points = k3d.points(positions=xspa_sag.astype(np.float32),point_size=2.7,shader='3d',color=0x3f6bc5) #, 0x6a329f, 0xc90076, 0xead1dc, 0xd5a6bd, 0xc27ba0, 0xa64d79, 0x741b47])
plt_points_cor = k3d.points(positions=xspa_cor.astype(np.float32),point_size=2.7,shader='3d',color=0xde49a1)
plt_points_ax = k3d.points(positions=xspa_ax.astype(np.float32),point_size=2.7,shader='3d',color=0xead1dc)

plt_points_ori = k3d.points(positions=np.array([xspa_sag[0], xspa_cor[1], xspa_ax[5]]).astype(np.float32),
                        point_size=4,
                        shader='3d',
                        color=0xc27ba0)
plt_points_z = k3d.points(positions=np.array([0., 0.,0.]).astype(np.float32),
                        point_size=15,
                        shader='3d',
                        color=0xc27ba0)
plot = k3d.plot()
#plot += plt_vectors
plot += plt_points
plot += plt_points_cor
plot += plt_points_ax
plot += plt_points_z
plot += plt_points_ori
plot.display()

In [ ]:
def get_int(IJK, floorIJK, serie, newres):
    shapeX, shapeY, shapeZ = serie.shape 
    d= newres*3
    i= floorIJK[...,2].astype(np.int32)
    j= floorIJK[...,1].astype(np.int32)
    k= floorIJK[...,0].astype(np.int32)
    #print(i.shape, i)

    di= IJK[..., 2]-i
    dj= IJK[..., 1]-j
    dk= IJK[..., 0]-k

    condition_i = (i >= 0) & (i < shapeX - 3)
    condition_j = (j >= 0) & (j < shapeY - 3)
    condition_k = (k >= 0) & (k < shapeZ - 3)
    
    condition_di = (di <= d)
    condition_dj = (dj <= d)
    condition_dk = (dk <= d)

    # Combine the conditions for all three dimensions using logical AND
    condition_all = condition_i & condition_j & condition_k & condition_di & condition_dj & condition_dk

    # Create the new array (bounds) with values 1 where the condition is True, and 0 where it is False
    bounds = np.where(condition_all, 1, 0)

    i=i[np.where(bounds==1)]
    j=j[np.where(bounds==1)]
    k=k[np.where(bounds==1)]

    di=di[np.where(bounds==1)]
    dj=dj[np.where(bounds==1)]
    dk=dk[np.where(bounds==1)]
    
    # Create an array of zeros with the same shape as bounds
    new_vol = np.zeros((bounds.shape))
    new_vol[:] =np.nan
    # Apply the operation where bounds is equal to 1


    """
    new_vol[bounds == 1]=(((1-di)*(1-dj)*(1-dk)*serie[i, j, k])+
              ((1-di)*(1-dj)*(dk)*serie[i, j, k+1]) +
              ((1-di)*(dj)*(1-dk)*serie[i, j+1, k])+
              ((di)*(1-dj)*(1-dk)*serie[i+1, j, k])+
              ((di)*(1-dj)*(dk)*serie[i+1, j, k+1])+
              ((di)*(dj)*(1-dk)*serie[i+1, j+1, k])+
              ((1-di)*(dj)*(dk)*serie[i, j+1, k+1])+
              ((di)*(dj)*(dk)*serie[i+1, j+1, k+1]))
    """
    #print((serie[i, j, k]/np.sqrt(((1-di)**2)+((1-dj)**2) +((1-dk)**2))))
    
    new_vol[bounds == 1]= (
        ((serie[i, j, k]/np.sqrt(((1-di)**2)+((1-dj)**2) +((1-dk)**2)))+
        (serie[i, j, k+1]/np.sqrt(((1-di)**2)+((1-dj)**2)+((dk)**2)))+
        (serie[i, j+1, k]/np.sqrt(((1-di)**2)+(dj**2)+((1-dk)**2)))+
        (serie[i+1, j, k]/np.sqrt((di**2)+((1-dj)**2)+((1-dk)**2)))+
        (serie[i+1, j, k+1]/np.sqrt((di**2)+((1-dj)**2)+(dk**2)))+
        (serie[i+1, j+1, k]/np.sqrt((di**2)+(dj**2)+((1-dk)**2)))+
        (serie[i, j+1, k+1]/np.sqrt(((1-di)**2)+(dj**2)+(dk**2)))+
        (serie[i+1, j+1, k+1]/np.sqrt((di**2)+(dj**2)+(dk**2))))/(
            ((1-di)**2+(1-dj)**2 +(1-dk)**2 )+ 
            ((1-di)**2+(1-dj)**2+(dk)**2)+
            ((1-di)**2+(dj)**2+(1-dk)**2)+
            ((di)**2+(1-dj)**2+(1-dk)**2)+ 
            ((di)**2+(1-dj)**2+(dk)**2) + 
            ((di)**2+(dj)**2+(1-dk)**2) +
            ((1-di)**2+(dj)**2+(dk)**2) +
            ((di)**2+(dj)**2+(dk)**2)))
              
    return new_vol


In [ ]:
s=160 #128 crop

I=np.arange(int((sizeX/2)-s),int((sizeX/2)+s))
J=np.arange(int((sizeY/2)-s),int((sizeY/2)+s))
K=np.arange(int((sizeZ/2)-s),int((sizeZ/2)+s))
#np.dot(A_new, np.array([i,j,k]))
print(I)
A_new=np.dot(dir_new, new_spacing)

ii, jj, kk = np.meshgrid(I,J,K, indexing='ij')
I012 = np.zeros((ii.shape[0], jj.shape[1], kk.shape[2], 3))
I012[:,:,:,0] = ii
I012[:,:,:,1] = jj
I012[:,:,:,2] = kk
print(ii.shape, jj.shape, kk.shape)
print(sizeX, sizeY, sizeZ)

XYZ=new_origin + np.einsum('ijkl,lm->ijkm', I012, A_new)
print(XYZ.shape)

# Calculate XYZ - origin_sag first
XYZ_minus_origin_sag = XYZ - origin_sag.reshape(1, 3)
XYZ_minus_origin_cor = XYZ - origin_cor.reshape(1, 3)
XYZ_minus_origin_ax = XYZ - origin_ax.reshape(1, 3)
invA_sag=np.linalg.inv(A_sag)
invA_cor=np.linalg.inv(A_cor)
invA_ax=np.linalg.inv(A_ax)

IJK_sag=np.matmul(XYZ_minus_origin_sag, invA_sag.T)
IJK_cor=np.matmul(XYZ_minus_origin_cor, invA_cor.T)
IJK_ax =np.matmul(XYZ_minus_origin_ax, invA_ax.T)

floor_IJK_sag=np.floor(IJK_sag)
floor_IJK_cor=np.floor(IJK_cor)
floor_IJK_ax =np.floor(IJK_ax)

sag_i=get_int(IJK_sag, floor_IJK_sag, m_sag, newres)
cor_i=get_int(IJK_cor, floor_IJK_cor, m_cor, newres)
ax_i=get_int(IJK_ax, floor_IJK_ax, m_ax, newres)
combined_vol = np.nanmean([sag_i, cor_i, ax_i], axis=0)


In [ ]:
print(np.sum(np.isnan(combined_vol)))
print(np.sum(~np.isnan(combined_vol)))
combined_vol=np.nan_to_num(combined_vol, nan=0).astype(np.int32)
print(np.sum(np.isnan(combined_vol)))
print(combined_vol.shape)

In [ ]:
plt_vol= k3d.volume(combined_vol.astype(np.float32), alpha_coef=75,color_map=matplotlib_color_maps.Turbo)
plot = k3d.plot()
plot += plt_vol

plot.display()

In [ ]:
print(ax_image.GetSpacing(), (newres, newres, newres) )
itk_view = itk_view_from_simpleitk(combined_vol.astype(np.float32), (newres, newres, newres), dir_new, new_origin)
#itk_view = itk_view_from_simpleitk(m_ax, ax_image.GetSpacing(), dir_ax, origin_ax)
itkwidgets.view(itk_view)

In [ ]:
# Converting back to SimpleITK (assumes we didn't move the image in space as we copy the information from the original)
filename='SUPERTESTVOL04_01TEST.nii.gz'
nv=combined_vol.astype(np.uint16)
result_image = sitk.Image(list(nv.shape), sitk.sitkUInt16)
result_image = sitk.GetImageFromArray(nv)
result_image.SetSpacing((newres, newres, newres))
result_image.SetOrigin(new_origin)
result_image.SetDirection(tuple(dir_new.flatten()))
    
# write the image
sitk.WriteImage(result_image, filename)

In [ ]:
et = time.time()
# get the execution time
elapsed_time = (et - st)/60
print('Execution time:', elapsed_time, 'minutes')